# Analyse de volumétrie DuckDB — version corrigée

Ce notebook :
- convertit le CSV en Parquet si besoin ;
- enlève la colonne parasite `column56` avant l'analyse ;
- calcule la volumétrie globale ;
- mesure les nulls par colonne ;
- mesure les doublons **exacts de lignes** avec une requête corrigée ;
- montre **des exemples de lignes dupliquées** pour voir si les doublons portent seulement des `NULL` ou aussi d'autres valeurs ;
- relie les colonnes problématiques au dictionnaire métier quand c'est utile.

Point important :
- un **null** = une valeur manquante dans une colonne ;
- un **doublon** = deux lignes ou plus strictement identiques sur **toutes** les colonnes analysées ;
- ce n'est donc **pas du tout la même chose**.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

csv_file = Path("A202505.csv")
parquet_file = Path("A202505.parquet")
clean_parquet_file = Path("A202505_clean.parquet")
dictionary_file = Path("2024_descriptif-variables_open-damir-base-complete.xlsx")

csv_delim = ";"
csv_header = True

con = duckdb.connect()

print("DuckDB version :", duckdb.__version__)
print("CSV :", csv_file.resolve())
print("Parquet source :", parquet_file.resolve())
print("Parquet nettoyé :", clean_parquet_file.resolve())
print("Dictionnaire présent :", dictionary_file.exists())

DuckDB version : 1.5.1
CSV : /Users/la_brioche/kDrive/Master M1 Drive/S2/Projet FIN/A202505.csv
Parquet source : /Users/la_brioche/kDrive/Master M1 Drive/S2/Projet FIN/A202505.parquet
Parquet nettoyé : /Users/la_brioche/kDrive/Master M1 Drive/S2/Projet FIN/A202505_clean.parquet
Dictionnaire présent : False


## 1. Conversion CSV → Parquet

Cette cellule ne reconvertit pas le CSV si le Parquet existe déjà.

In [2]:
if parquet_file.exists():
    print(f"Le fichier Parquet existe déjà : {parquet_file}")
else:
    if not csv_file.exists():
        raise FileNotFoundError(f"Fichier introuvable : {csv_file}")

    con.execute(f'''
        COPY (
            SELECT *
            FROM read_csv_auto(
                '{csv_file.as_posix()}',
                delim='{csv_delim}',
                header={str(csv_header).lower()}
            )
        )
        TO '{parquet_file.as_posix()}' (FORMAT PARQUET)
    ''')

    print(f"Conversion terminée : {parquet_file}")

Le fichier Parquet existe déjà : A202505.parquet


## 2. Nettoyage minimal avant analyse

`column56` est supprimée avant l'analyse :
- elle est entièrement vide ;
- elle vient très probablement d'un séparateur final en trop ou d'un export CSV imparfait.

In [4]:
schema_source_df = con.execute(f"DESCRIBE SELECT * FROM '{parquet_file.as_posix()}'").df()
source_columns = schema_source_df["column_name"].tolist()

if "column56" in source_columns:
    con.execute(f'''
        COPY (
            SELECT * EXCLUDE (column56)
            FROM '{parquet_file.as_posix()}'
        )
        TO '{clean_parquet_file.as_posix()}' (FORMAT PARQUET)
    ''')
    print("Parquet nettoyé créé sans column56.")
else:
    con.execute(f'''
        COPY (
            SELECT *
            FROM '{parquet_file.as_posix()}'
        )
        TO '{clean_parquet_file.as_posix()}' (FORMAT PARQUET)
    ''')
    print("Aucune column56 détectée : copie simple du parquet source.")

table_ref = clean_parquet_file.as_posix()
print("Table analysée :", table_ref)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet nettoyé créé sans column56.
Table analysée : A202505_clean.parquet


## 3. Aperçu rapide du jeu de données analysé

In [5]:
preview_df = con.execute(f"SELECT * FROM '{table_ref}' LIMIT 10").df()
preview_df

,FLX_ANN_MOI,ORG_CLE_REG,AGE_BEN_SNDS,BEN_RES_REG,BEN_CMU_TOP,BEN_QLT_COD,BEN_SEX_COD,DDP_SPE_COD,ETE_CAT_SNDS,ETE_REG_COD,...,PSE_ACT_CAT,PSE_SPE_SNDS,PSE_STJ_SNDS,PRE_INS_REG,PSP_ACT_SNDS,PSP_ACT_CAT,PSP_SPE_SNDS,PSP_STJ_SNDS,TOP_PS5_TRG,ETB_DCS_MCO
0,202505,76,70,76,1,1,2,0,1101,76,...,0,0,2,99,0,0,1,2,1,M
1,202505,11,70,11,0,1,1,121,9999,99,...,7,0,1,11,0,1,1,1,1,Z
2,202505,11,30,11,0,1,2,45,1107,11,...,0,0,2,99,0,99,0,9,1,S
3,202505,52,50,52,0,1,2,121,9999,99,...,3,0,1,99,0,0,42,2,1,Z
4,202505,11,80,11,1,1,2,121,9999,99,...,7,0,1,99,0,0,1,2,1,Z
5,202505,76,70,76,0,1,1,121,9999,99,...,6,0,1,76,0,1,1,1,1,Z
6,202505,11,70,11,0,1,2,0,1102,11,...,0,4,2,99,0,0,1,2,1,C
7,202505,32,80,32,0,1,2,121,9999,99,...,1,1,1,32,0,1,1,1,1,Z
8,202505,99,80,99,0,1,2,121,9999,99,...,2,0,1,99,0,0,31,2,1,Z
9,202505,11,80,76,0,1,2,121,9999,99,...,6,0,1,76,0,1,12,1,1,Z


In [6]:
schema_df = con.execute(f"DESCRIBE SELECT * FROM '{table_ref}'").df()
schema_df

,column_name,column_type,null,key,default,extra
0,FLX_ANN_MOI,BIGINT,YES,None,None,None
1,ORG_CLE_REG,BIGINT,YES,None,None,None
2,AGE_BEN_SNDS,BIGINT,YES,None,None,None
3,BEN_RES_REG,BIGINT,YES,None,None,None
4,BEN_CMU_TOP,BIGINT,YES,None,None,None
5,BEN_QLT_COD,BIGINT,YES,None,None,None
6,BEN_SEX_COD,BIGINT,YES,None,None,None
7,DDP_SPE_COD,BIGINT,YES,None,None,None
8,ETE_CAT_SNDS,BIGINT,YES,None,None,None
9,ETE_REG_COD,BIGINT,YES,None,None,None


## 4. Métadonnées globales

On calcule :
- le nombre de lignes ;
- le nombre de colonnes ;
- le nombre total de cellules ;
- le nombre total de cellules nulles ;
- le pourcentage global de nulls.

In [7]:
table_name = "volumetrie_tmp"
con.execute(f"CREATE OR REPLACE TEMP VIEW {table_name} AS SELECT * FROM '{table_ref}'")

columns_df = con.execute(f"DESCRIBE SELECT * FROM {table_name}").df()
columns = columns_df["column_name"].tolist()
column_types = dict(zip(columns_df["column_name"], columns_df["column_type"]))

nb_colonnes = len(columns)
nb_lignes = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]

null_exprs = [f'SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS "{col}"' for col in columns]
nulls_wide_df = con.execute(f"SELECT {', '.join(null_exprs)} FROM {table_name}").df()

nulls_long_df = (
    nulls_wide_df.T
    .reset_index()
    .rename(columns={"index": "column_name", 0: "null_count"})
)

nulls_long_df["null_count"] = nulls_long_df["null_count"].astype("int64")
nulls_long_df["null_pct"] = (nulls_long_df["null_count"] / nb_lignes * 100).round(4)
nulls_long_df["column_type"] = nulls_long_df["column_name"].map(column_types)
nulls_long_df = nulls_long_df[["column_name", "column_type", "null_count", "null_pct"]].sort_values(
    by=["null_count", "column_name"], ascending=[False, True]
).reset_index(drop=True)

nb_null_total = int(nulls_long_df["null_count"].sum())
nb_cellules = int(nb_lignes * nb_colonnes)
pct_null_global = round((nb_null_total / nb_cellules * 100), 4) if nb_cellules else 0.0

global_df = pd.DataFrame([{
    "nb_lignes": nb_lignes,
    "nb_colonnes": nb_colonnes,
    "nb_cellules_total": nb_cellules,
    "nb_null_total": nb_null_total,
    "pct_null_global": pct_null_global
}])

global_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,nb_lignes,nb_colonnes,nb_cellules_total,nb_null_total,pct_null_global
0,34167564,56,1913383584,7908586,0.4133


## 5. Nulls par colonne

Ce tableau permet d'identifier les colonnes réellement problématiques.
- Un taux de null élevé peut signaler un problème technique.
- Mais il peut aussi être **normal métier** si le dictionnaire explique que la variable n'est pas alimentée sur tout le périmètre.

In [8]:
nulls_long_df

,column_name,column_type,null_count,null_pct
0,PRS_ACT_NBR,BIGINT,4120235,12.0589
1,FLT_ACT_NBR,BIGINT,3788351,11.0876
2,AGE_BEN_SNDS,BIGINT,0,0.0000
3,ASU_NAT,BIGINT,0,0.0000
4,ATT_NAT,BIGINT,0,0.0000
5,BEN_CMU_TOP,BIGINT,0,0.0000
6,BEN_QLT_COD,BIGINT,0,0.0000
7,BEN_RES_REG,BIGINT,0,0.0000
8,BEN_SEX_COD,BIGINT,0,0.0000
9,CPL_COD,BIGINT,0,0.0000


In [9]:
nulls_with_dict_df = (
    nulls_long_df[nulls_long_df["null_count"] > 0]
    .merge(dict_df[["column_name", "label", "family", "description"]], on="column_name", how="left")
)

nulls_with_dict_df

,column_name,column_type,null_count,null_pct,label,family,description
0,PRS_ACT_NBR,BIGINT,4120235,12.0589,NaN,NaN,NaN
1,FLT_ACT_NBR,BIGINT,3788351,11.0876,NaN,NaN,NaN


### Lecture utile des nulls observés

Les deux colonnes de dénombrement les plus sensibles sont généralement :

- **`PRS_ACT_NBR`** : dénombrement brut.  
  Le dictionnaire précise qu'il faut filtrer correctement le type de remboursement et qu'il faut **privilégier la quantité au dénombrement**, car le dénombrement n'est pas remonté pour l'ensemble des régimes.

- **`FLT_ACT_NBR`** : dénombrement préfiltré.  
  Cette variable évite déjà le double comptage lié au type de remboursement, mais elle reste partiellement renseignée car le dénombrement n'est pas disponible sur tout le périmètre.

Donc :
- beaucoup de nulls sur ces colonnes ne veulent **pas forcément** dire que la donnée est mauvaise ;
- cela peut simplement traduire une **limite de couverture métier** ;
- pour une analyse globale, il faut souvent préférer les variables de **quantité** (`*_QTE`) aux variables de **dénombrement** (`*_NBR`).

## 6. Doublons exacts de lignes — définition correcte

Ici, un doublon signifie :

> deux lignes ou plus strictement identiques sur **toutes** les colonnes analysées.

Important :
- les `NULL` comptent dans la comparaison ;
- deux lignes sont doublons si elles ont les **mêmes valeurs partout** et les `NULL` aux **mêmes endroits** ;

Exemple :
- ligne A = `(1, NULL, 202505)`  
- ligne B = `(1, NULL, 202505)`  
  → doublon exact

- ligne A = `(1, NULL, 202505)`  
- ligne B = `(1, 0, 202505)`  
  → **pas** un doublon

In [10]:
duplicates_df = con.execute(f'''
WITH groupes AS (
    SELECT *, COUNT(*) AS cnt
    FROM {table_name}
    GROUP BY ALL
    HAVING COUNT(*) > 1
)
SELECT
    COUNT(*) AS nb_groupes_doublons,
    COALESCE(SUM(cnt - 1), 0) AS nb_lignes_dupliquees,
    COALESCE(SUM(cnt), 0) AS nb_lignes_concernees
FROM groupes
''').df()

duplicates_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,nb_groupes_doublons,nb_lignes_dupliquees,nb_lignes_concernees
0,0,0.0,0.0


## 7. Exemples concrets de doublons

Cette cellule affiche des groupes de lignes dupliquées avec :
- `nb_occurrences` = nombre de fois où la ligne identique apparaît ;
- `nb_nulls_ligne` = nombre de colonnes nulles dans cette ligne.

In [11]:
null_count_expr = " + ".join([f'CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END' for col in columns])

duplicate_examples_df = con.execute(f'''
SELECT
    *,
    COUNT(*) AS nb_occurrences,
    ({null_count_expr}) AS nb_nulls_ligne
FROM {table_name}
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY nb_occurrences DESC, nb_nulls_ligne DESC
LIMIT 20
''').df()

duplicate_examples_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,FLX_ANN_MOI,ORG_CLE_REG,AGE_BEN_SNDS,BEN_RES_REG,BEN_CMU_TOP,BEN_QLT_COD,BEN_SEX_COD,DDP_SPE_COD,ETE_CAT_SNDS,ETE_REG_COD,...,PSE_STJ_SNDS,PRE_INS_REG,PSP_ACT_SNDS,PSP_ACT_CAT,PSP_SPE_SNDS,PSP_STJ_SNDS,TOP_PS5_TRG,ETB_DCS_MCO,nb_occurrences,nb_nulls_ligne


## 8. Analyse détaillée par colonne

Pour chaque colonne :
- type ;
- nombre de valeurs non nulles ;
- nombre de nulls ;
- pourcentage de nulls ;
- nombre de valeurs distinctes non nulles.

Attention :
- `COUNT(DISTINCT ...)` peut être coûteux sur un très gros fichier ;
- sur 6 Go, cette cellule peut prendre du temps.

In [13]:
detail_rows = []

for col in columns:
    q = f'''
    SELECT
        COUNT(*) AS row_count,
        COUNT("{col}") AS non_null_count,
        SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS null_count,
        COUNT(DISTINCT "{col}") AS distinct_non_null_count
    FROM {table_name}
    '''
    row = con.execute(q).fetchone()
    row_count, non_null_count, null_count, distinct_non_null_count = row
    detail_rows.append({
        "column_name": col,
        "column_type": column_types[col],
        "row_count": int(row_count),
        "non_null_count": int(non_null_count),
        "null_count": int(null_count),
        "null_pct": round((null_count / row_count * 100), 4) if row_count else 0.0,
        "distinct_non_null_count": int(distinct_non_null_count)
    })

detail_df = pd.DataFrame(detail_rows).sort_values(
    by=["null_count", "column_name"], ascending=[False, True]
).reset_index(drop=True)

detail_df

,column_name,column_type,row_count,non_null_count,null_count,null_pct,distinct_non_null_count
0,PRS_ACT_NBR,BIGINT,34167564,30047329,4120235,12.0589,18392
1,FLT_ACT_NBR,BIGINT,34167564,30379213,3788351,11.0876,17946
2,AGE_BEN_SNDS,BIGINT,34167564,34167564,0,0.0000,9
3,ASU_NAT,BIGINT,34167564,34167564,0,0.0000,11
4,ATT_NAT,BIGINT,34167564,34167564,0,0.0000,4
5,BEN_CMU_TOP,BIGINT,34167564,34167564,0,0.0000,4
6,BEN_QLT_COD,BIGINT,34167564,34167564,0,0.0000,5
7,BEN_RES_REG,BIGINT,34167564,34167564,0,0.0000,14
8,BEN_SEX_COD,BIGINT,34167564,34167564,0,0.0000,3
9,CPL_COD,BIGINT,34167564,34167564,0,0.0000,4


## 9. Résumé final consolidé

In [14]:
resume_df = global_df.copy()
resume_df["nb_groupes_doublons"] = int(duplicates_df.loc[0, "nb_groupes_doublons"])
resume_df["nb_lignes_dupliquees"] = int(duplicates_df.loc[0, "nb_lignes_dupliquees"])
resume_df["nb_lignes_concernees_par_doublons"] = int(duplicates_df.loc[0, "nb_lignes_concernees"])

resume_df

,nb_lignes,nb_colonnes,nb_cellules_total,nb_null_total,pct_null_global,nb_groupes_doublons,nb_lignes_dupliquees,nb_lignes_concernees_par_doublons
0,34167564,56,1913383584,7908586,0.4133,0,0,0
